# Jetson 車道模型：在 Colab 接續訓練

1. 在 Google 雲端硬碟的「我的雲端硬碟」根目錄上傳 `car_training_colab.zip`。
2. 在 Colab 選「執行階段 → 變更執行階段類型 → GPU」。
3. 由上往下依序執行每個程式格。第一次連接雲端硬碟時，請用上傳 ZIP 的 Google 帳號授權。
4. 最後一格會把新模型存到雲端硬碟根目錄。下載後先在 Jetson 驗證能載入，再試車。

這份筆記本接續 ZIP 裡的舊模型，訓練 10 輪，並保留原始模型。Colab 的免費 GPU 與執行時間依當時供應情況而定。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import torch, torchvision

archive = Path('/content/drive/MyDrive/car_training_colab.zip')
workdir = Path('/content/car_training')
assert archive.is_file(), '找不到 ZIP；請先把 car_training_colab.zip 上傳到我的雲端硬碟根目錄'
assert torch.cuda.is_available(), '目前沒有 GPU；請在「執行階段 → 變更執行階段類型」選 GPU'
workdir.mkdir(exist_ok=True)
with ZipFile(archive) as package:
    package.extractall(workdir)
for name in ('train_model_terminal.py', 'best_steering_model_xy.pth'):
    assert (workdir / name).is_file(), 'ZIP 缺少 ' + name
count = len(list((workdir / 'dataset_xy').glob('*.jpg')))
assert count >= 20, '沒有找到足夠的標註照片'
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0), 'JPG:', count)


## 接續訓練

這格完成時會顯示 `Training complete`。訓練期間請保持 Colab 執行階段連線；每一輪會顯示 `Epoch ... train=... validation=...`。


In [ ]:
import subprocess, sys

output = workdir / 'best_steering_model_xy_colab.pth'
command = [
    sys.executable, 'train_model_terminal.py',
    '--data', 'dataset_xy',
    '--resume', 'best_steering_model_xy.pth',
    '--output', output.name,
    '--epochs', '10',
    '--batch-size', '8',
]
subprocess.run(command, cwd=workdir, check=True)
assert output.is_file(), '訓練沒有產出模型'


## 匯出供 Jetson 測試的模型

Jetson 上使用較舊的 PyTorch。這格把權重另存為舊版序列化格式，然後複製到 Google 雲端硬碟。格式轉換有助於相容，仍須在 Jetson 實際載入確認。


In [ ]:
import shutil

state = torch.load(output, map_location='cpu', weights_only=True)
legacy = workdir / 'best_steering_model_xy_colab_jetson.pth'
torch.save(state, legacy, _use_new_zipfile_serialization=False)
destination = Path('/content/drive/MyDrive') / legacy.name
shutil.copy2(legacy, destination)
print('完成：', destination, '大小 MB：', round(destination.stat().st_size / 1_000_000, 1))
